In [1]:
import os
import time
from pathlib import Path
import polars as pl
import duckdb

In [2]:
DATA_PATH = "../data/raw/chicago_crimes.csv"
print(f"Ukuran file: {Path(DATA_PATH).stat().st_size / 1024**2:.1f} MB")

Ukuran file: 1825.6 MB


In [3]:
lf = pl.scan_csv(
    DATA_PATH,
    try_parse_dates=True,
    schema_overrides={
        "IUCR": pl.Utf8,
        "FBI Code": pl.Utf8,
        "Case Number": pl.Utf8,
    },
    infer_schema_length=10000,
    ignore_errors=True,
)
print(lf.collect_schema())

Schema({'ID': Int64, 'Case Number': String, 'Date': String, 'Block': String, 'IUCR': String, 'Primary Type': String, 'Description': String, 'Location Description': String, 'Arrest': Boolean, 'Domestic': Boolean, 'Beat': Int64, 'District': Int64, 'Ward': Int64, 'Community Area': Int64, 'FBI Code': String, 'X Coordinate': Int64, 'Y Coordinate': Int64, 'Year': Int64, 'Updated On': String, 'Latitude': Float64, 'Longitude': Float64, 'Location': String})


In [4]:
n_rows = lf.select(pl.len()).collect().item()
print(f"Jumlah baris: {n_rows:,}")

Jumlah baris: 8,104,658


In [5]:
con = duckdb.connect()
con.sql(f"SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}', ignore_errors=true)").pl()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""ID""","""BIGINT""","""634""","""13525542""",8692979,"""7272202.810450484""","""3642948.1631512283""","""3901046""","""7302679""","""10508633""",8104658,0.00
"""Case Number""","""VARCHAR""","""01G050460""","""ZZZ199957""",9812079,null,null,null,null,null,8104658,0.00
"""Date""","""TIMESTAMP""","""2001-01-01 00:00:00""","""2024-07-04 00:00:00""",3775371,"""2010-12-22 13:37:17.028808""",null,"""2005-03-17 18:39:16.220325""","""2010-01-05 20:15:30.010639""","""2016-05-09 20:04:06.697114""",8104658,0.00
"""Block""","""VARCHAR""","""0000X E 100 PL""","""XX UNKNOWN""",54224,null,null,null,null,null,8104658,0.00
"""IUCR""","""VARCHAR""","""0110""","""9901""",484,null,null,null,null,null,8104658,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""Year""","""BIGINT""","""2001""","""2024""",26,"""2010.4736085100692""","""6.670175334272892""","""2005""","""2009""","""2016""",8104658,0.00
"""Updated On""","""TIMESTAMP""","""2006-03-31 22:03:38""","""2024-07-11 15:45:11""",6081,"""2018-08-22 23:40:18.115331""",null,"""2018-02-10 15:50:01""","""2018-02-28 15:56:25""","""2018-02-28 23:13:30.693857""",8104658,0.00
"""Latitude""","""DOUBLE""","""36.619446395""","""42.022910333""",864066,"""41.84237384933105""","""0.0893371230982746""","""41.7688706358803""","""41.85642688397502""","""41.906841571306124""",8104658,1.11


In [6]:
lf.null_count().collect()

ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,13257,0,0,0,47,614831,613455,0,89642,89642,0,0,89642,89642,89642


In [7]:
lf. with_columns(
    pl.col("Date").str.to_datetime("%m/%d/%Y %I:%M:%S %p", strict=False).alias("parsed_date")
).select(
    pl.col("parsed_date").min().alias("tanggal_awal"),
    pl.col("parsed_date").max().alias("tanggal_akhir"),
    pl.col("Year").min().alias("tahun_awal"),
    pl.col("Year").max().alias("tahun_akhir"),
    pl.col("Latitude").min().alias("lat_min"),
    pl.col("Latitude").max().alias("lat_max"),
    pl.col("Longitude").min().alias("lon_min"),
    pl.col("Longitude").max().alias("lon_max"),
).collect()

tanggal_awal,tanggal_akhir,tahun_awal,tahun_akhir,lat_min,lat_max,lon_min,lon_max
datetime[μs],datetime[μs],i64,i64,f64,f64,f64,f64
2001-01-01 00:00:00,2024-07-04 00:00:00,2001,2024,36.619446,42.02291,-91.686566,-87.524529


In [8]:
n_unik = lf.unique().select(pl.len()).collect().item()
n_id_unik = lf.select(pl.col("ID").n_unique()).collect().item()

print(f"Baris duplikat penuh : {n_rows - n_unik:,}")
print(f"Duplikat kolom ID    : {n_rows - n_id_unik:,}")

Baris duplikat penuh : 0
Duplikat kolom ID    : 0


In [9]:
lf.group_by("Primary Type").agg(
    pl.len().alias("jumlah_kasus"),
    (pl.col("Arrest").mean() * 100).round(2).alias("persen_arrest"),
).sort("jumlah_kasus", descending=True).head(10).collect()

Primary Type,jumlah_kasus,persen_arrest
str,u32,f64
"""THEFT""",1713138,10.95
"""BATTERY""",1478402,21.78
"""CRIMINAL DAMAGE""",923178,6.58
"""NARCOTICS""",754665,99.35
"""ASSAULT""",535559,20.66
"""OTHER OFFENSE""",503568,17.56
"""BURGLARY""",433444,5.71
"""MOTOR VEHICLE THEFT""",406982,7.83
"""DECEPTIVE PRACTICE""",366611,12.86
